# All of Us slide panels for the ASHG 2026 deck — standalone

The right-hand panel of each UK Biobank | All of Us pair (deck slides 26–29),
drawn to match the UKB panel beside it. **Standalone**: upload this one file
and run it. It needs numpy, pandas and matplotlib, the merged bin tables, and
nothing from the repo.

**Input:** `<WORK_DIR>/<phenotype>__<transform>__<covset>_merged.full.tsv` from
`08_accumulate.ipynb`, and the bins file it was merged with. Only the full-data
sums are used — no jackknife, since no interval is drawn.

**Output** (`OUT_DIR`): for each trait, the UKB panels' three build steps with
identical axes — `points_only`, `points_h2`, `with_fits` — as PNG (300 dpi) and
PDF at 5.55 × 3.30 in (insert at 100%), plus the plotted values under `tables/`.

What goes in:

- **Points**: `noPO` bin means (`other` + `FS`); **parent-offspring** bins as
  triangles, in no fit. Bins with fewer than `MIN_PAIRS_PER_BIN` pairs are not
  drawn. Points stop at 0.6, where the UKB panels stop. The merge's 0.001-wide
  bins below 0.02 are pooled to UKB's 0.005 for drawing (bin sums add, so a
  pooled mean is exact); the fits always use the merge's own bins.
- **Fits**, on every populated `noPO` bin in range, by pair-weighted least
  squares — the same fits `he_estimators` makes:

| legend | fitted on | design | `he_estimators` estimator |
|---|---|---|---|
| FS | 0.4–0.6 | slope + intercept | `b2_FS` |
| HS | 0.2–0.3 (UKB's HS window) | slope + intercept | none |
| Rel | 0.02–0.7 | slope + intercept | `h2_Rel` |
| OneSlopeOffsets | 0.09–0.7, bands at 0.18 / 0.36 | one slope, an intercept per band | `h2_OneSlopeOffsets` |

- **Dashed line**: `y = h2_Unrel · a_ij`, the slope from unrelated pairs
  (`a_ij < 0.02`, through the origin), labelled h²_SNP (Unrel).
- **y axis**: the paired UKB panel's range wherever every point and fit fits
  inside it; otherwise widened on that side only, and reported.

The look — size, Tenorite, margins, legend, line weights, colours — is copied
from `aou_slide_figures.py` in the deck's `UKB figures/` folder, which was
checked pixel-identical to the UKB panels. If that file changes, update the
style cell to match.

**Fonts:** put the Tenorite `.ttf` files in `FONT_DIR` (default `~/fonts`).
Without them the panels fall back to DejaVu Sans, and the style cell says so.

**Leaving the Workbench:** figures and tables are aggregate and every drawn bin
has at least `MIN_PAIRS_PER_BIN` pairs, but they still go through the AoU
dissemination rules before they reach the deck.

## Settings

In [ ]:
import glob
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.lines import Line2D
from IPython.display import Image, display

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"
WORK_DIR    = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")   # the merged tables
BINS_FILE   = f"{WORK_DIR}/bins_wide.txt"                         # the bins they were merged with
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
OUT_DIR  = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}/07_estimators/slide_panels"
FONT_DIR = os.path.expanduser("~/fonts")                           # Tenorite .ttf files
PLOTS_MIRROR = os.path.expanduser(f"~/plots/{RUN_DIR}/slide_panels")

TRANSFORM, COVSET = "invnorm", "base_pcs"   # the reference combo, as in 11 and 11b

MIN_PAIRS_PER_BIN = 50     # display only; every fit uses every populated bin
SLIDE_MAX_A = 0.6          # points and lines stop where the UKB panels' do
DISPLAY_BIN = 0.005        # UKB's bin width; narrower display bins are pooled up to it
PO_XOFF = 0.004            # PO and FS share bins; nudge so a triangle cannot hide a circle

# Model constants -- the values in src/he_estimators.py
U_HI, R_HI = 0.02, 0.7
FS_RANGE = (0.4, 0.6)
HS_WINDOW = (0.2, 0.3)     # UKB's HS* window; not he_estimators' HS_RANGE (0.2-0.4)
OFFSET_BANDS = {"band1": (0.09, 0.18), "band2": (0.18, 0.36), "band3": (0.36, R_HI)}
BAND_LO = min(lo for lo, _ in OFFSET_BANDS.values())
NOPO = ["other", "FS"]

assert os.path.isfile(BINS_FILE), f"missing {BINS_FILE} — see 08_accumulate.ipynb"
EDGES = np.loadtxt(BINS_FILE)          # (lo, hi) per bin, in bin_index order
MID = EDGES.mean(axis=1)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(MID)} bins | {TRANSFORM} / {COVSET}\nout: {OUT_DIR}")

## Slide style

Copied from `aou_slide_figures.py` (deck folder `UKB figures/`), with one
addition: `panel()` takes a fixed `ylim`. Unrel is dropped from the fits, as
on the UKB panels — here it is the dashed line instead.

In [ ]:
SIZE = (5.55, 3.30)                       # inches, one panel of a UKB | All of Us pair
MARGINS = dict(L=0.78, B=0.58, R=0.12, T=0.62)
POINT, GRID, ZERO = "#333333", "#e5e5e5", "#b0b0b0"
FIT_LW, FIT_ALPHA = 1.8, 0.5
H2_STYLE = dict(color="#1a1a1a", lw=1.5, ls=(0, (4, 2.5)), alpha=0.9)
# the UKB panels say "(LDSC)"; this line is the unrelated-pair slope
H2_LABEL = r"$h^2_{\mathrm{SNP}}$ (Unrel)"
MODEL_COLORS = {
    "FS": "#be9b6e", "HS": "#5f9ea0", "RelOffsets": "#759672", "Rel": "#ce7b79",
    "OneSlopeOffsets": "#b5716d", "OneSlope": "#8e7dae",
}
RC = {
    "font.size": 12, "axes.titlesize": 14, "axes.labelsize": 13,
    "xtick.labelsize": 11.5, "ytick.labelsize": 11.5, "legend.fontsize": 11,
    "axes.linewidth": 0.9, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.width": 0.9, "ytick.major.width": 0.9,
    "pdf.fonttype": 42, "ps.fonttype": 42,
}

for _f in glob.glob(os.path.join(FONT_DIR, "**", "*.ttf"), recursive=True):
    fm.fontManager.addfont(_f)
FONT = "Tenorite" if "Tenorite" in {f.name for f in fm.fontManager.ttflist} else "DejaVu Sans"
plt.rcParams.update(RC)
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold"})
print(f"font: {FONT}" + ("" if FONT == "Tenorite" else
      f"  -- NOT the deck font; put the Tenorite .ttf files in {FONT_DIR} and re-run"))


def row_major(items, ncol):
    nrow = -(-len(items) // ncol)
    grid = [items[i * ncol:(i + 1) * ncol] for i in range(nrow)]
    return [row[c] for c in range(ncol) for row in grid if c < len(row)]


def panel(points, fits, variant, out, h2=None, ylim=None):
    w, h = SIZE
    m = MARGINS
    fig = plt.figure(figsize=SIZE)
    ax = fig.add_axes([m["L"] / w, m["B"] / h, (w - m["L"] - m["R"]) / w, (h - m["B"] - m["T"]) / h])
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.axhline(0, color=ZERO, lw=1.0, zorder=1)

    binned = points[points["group"] != "parent_offspring"]
    po = points[points["group"] == "parent_offspring"]
    ax.scatter(binned.x, binned.y, s=22, c=POINT, lw=0, zorder=4)
    if len(po):
        ax.scatter(po.x, po.y, s=30, c=POINT, marker="^", lw=0, zorder=4)

    unknown = set(fits.model) - set(MODEL_COLORS)
    if unknown:
        raise ValueError(f"no colour defined for model(s) {sorted(unknown)}")
    # every variant gets the with-fits axis ranges so build steps line up
    ax.update_datalim(list(zip(fits.x, fits.y)))
    ax.autoscale_view()
    if ylim is not None:
        ax.set_ylim(*ylim)
    if variant == "with_fits":
        for (model, _), seg in fits.groupby(["model", "segment"], sort=False):
            ax.plot(seg.x, seg.y, color=MODEL_COLORS[model], lw=FIT_LW, alpha=FIT_ALPHA, zorder=3,
                    solid_capstyle="butt")
    has_h2 = variant != "points_only" and h2 is not None
    if has_h2:
        x0, x1 = ax.get_xlim()
        ax.plot([x0, x1], [h2 * x0, h2 * x1], zorder=3.5, scalex=False, scaley=False, **H2_STYLE)

    ax.locator_params(axis="y", nbins=5)
    ax.set_xlabel(r"Genetic relatedness ($a_{ij}$)")
    ax.set_ylabel(r"Mean cross-product ($\tilde{y}_i\,\tilde{y}_j$)")

    hs = [Line2D([], [], ls="none", marker="o", ms=5, mfc=POINT, mec="none")]
    labs = ["Binned data"]
    if len(po):
        hs.append(Line2D([], [], ls="none", marker="^", ms=5.5, mfc=POINT, mec="none"))
        labs.append("Parent–offspring")
    if variant == "with_fits":
        for model in MODEL_COLORS:  # fixed legend order
            if model in set(fits.model):
                hs.append(Line2D([], [], color=MODEL_COLORS[model], lw=2.4)); labs.append(model)
    if has_h2:
        hs.append(Line2D([], [], **H2_STYLE)); labs.append(H2_LABEL)
    ax.legend(row_major(hs, 4), row_major(labs, 4), loc="lower left",
              bbox_to_anchor=(0.03, 1 - (0.62 - 0.1) / 3.30 + 0.02), bbox_transform=fig.transFigure,
              ncol=4, frameon=False, handlelength=1.5, handletextpad=0.45, columnspacing=0.9,
              labelspacing=0.3, borderaxespad=0, borderpad=0)
    fig.savefig(out + ".png", dpi=300)
    fig.savefig(out + ".pdf")
    plt.close(fig)

## Slides

One row per All of Us panel. `ukb_ylim` is the paired UKB panel's y range,
recomputed from `ukb/publication/extracted_fits.json` with the autoscale
`make_slide_figures.py` gives a wide panel (points plus the FS, HS, Rel and
OneSlopeOffsets fits, 5% margin). Set it to `None` to let a panel scale itself.
Alcohol pairs AUDIT-C here with UKB's *alcohol intake frequency*.

In [ ]:
SLIDES = pd.DataFrame([
    # phenotype,              file stem,             slide, ukb_ylim                 # paired UKB trait
    ("waist_circumference",   "Waist_circumference", 26, (-0.088687, 0.521584)),    # Waist circumference
    ("bmi",                   "Body_mass_index_BMI", 27, (-0.055967, 0.439216)),    # Body mass index (BMI)
    ("height",                "Standing_height",     28, (-0.040250, 0.689520)),    # Standing height
    ("alcohol_audit_c_score", "Alcohol_use_AUDIT_C", 29, (-0.034370, 0.354878)),    # Alcohol intake frequency
], columns=["phenotype", "stem", "slide", "ukb_ylim"])

SLIDES["full_tsv"] = [f"{WORK_DIR}/{p}__{TRANSFORM}__{COVSET}_merged.full.tsv"
                      for p in SLIDES["phenotype"]]
missing = SLIDES.loc[~SLIDES["full_tsv"].map(os.path.isfile), "phenotype"].tolist()
assert not missing, f"no {TRANSFORM} / {COVSET} merge in {WORK_DIR} for: {', '.join(missing)}"
print(SLIDES[["slide", "phenotype", "stem", "ukb_ylim"]].to_string(index=False))

## Bins and fits

`load_sums` reads the full-data sums and pair counts per class and bin.
Each fit is pair-weighted least squares on bin means over every populated
`noPO` bin in its range — identical to pair-level OLS with each pair at its
bin midpoint, and to what `he_estimators` fits. No fit crosses `a_ij = 0.02`.

In [ ]:
CLASSES = ["other", "FS", "PO"]


def load_sums(path):
    """{class: (sum, n)} arrays over the bin grid, from a *_merged.full.tsv."""
    f = pd.read_csv(path, sep="\t", usecols=["class", "bin_index", "full_sum", "full_n"])
    out = {}
    for c in CLASSES:
        s, n = np.zeros(len(MID)), np.zeros(len(MID))
        fc = f[f["class"] == c]
        k = fc["bin_index"].to_numpy() - 1
        s[k], n[k] = fc["full_sum"].to_numpy(), fc["full_n"].to_numpy()
        out[c] = (s, n)
    return out


def class_sums(sums, classes):
    return (sum(sums[c][0] for c in classes), sum(sums[c][1] for c in classes))


def band_ind(a):
    return np.column_stack([((a >= lo) & (a < hi)).astype(float) for lo, hi in OFFSET_BANDS.values()])


_origin = lambda a: a[:, None]
_slope_int = lambda a: np.c_[a, np.ones_like(a)]
# legend name -> (fitted ranges, design)
FITS = {
    "FS":              ([FS_RANGE],       _slope_int),
    "HS":              ([HS_WINDOW],      _slope_int),
    "Rel":             ([(U_HI, R_HI)],   _slope_int),
    "OneSlopeOffsets": ([(BAND_LO, R_HI)], lambda a: np.c_[a, band_ind(a)]),
}
UNREL = ([(-np.inf, U_HI)], _origin)    # the dashed line's slope

for _name, (_ranges, _) in {**FITS, "Unrel": UNREL}.items():
    _sides = {"u" if hi <= U_HI else "r" if lo >= U_HI else None for lo, hi in _ranges}
    assert None not in _sides and len(_sides) == 1, f"{_name} spans a_ij = {U_HI}"


def wls(X, m, n):
    """Pair-weighted least squares; None if under-determined."""
    if len(m) <= X.shape[1]:
        return None
    w = np.sqrt(n)
    Xw = X * w[:, None]
    if np.linalg.matrix_rank(Xw) < X.shape[1]:
        return None
    return np.linalg.lstsq(Xw, m * w, rcond=None)[0]


def fit(sums, ranges, design):
    """(beta, a of the bins fitted) on noPO, every populated bin in range."""
    s, n = class_sums(sums, NOPO)
    inside = np.zeros(len(MID), dtype=bool)
    for lo, hi in ranges:
        inside |= (MID >= lo) & (MID < hi)
    k = inside & (n > 0)
    a = MID[k]
    return (wls(design(a), s[k] / n[k], n[k]) if k.any() else None), a


def display_bins(sums, classes):
    """(mid, mean, n) for drawing: bins narrower than DISPLAY_BIN pooled into
    DISPLAY_BIN cells, wider bins kept. Sums add, so pooled means are exact."""
    s, n = class_sums(sums, classes)
    lo, hi = np.rint(EDGES[:, 0] * 1000).astype(int), np.rint(EDGES[:, 1] * 1000).astype(int)
    d = int(round(DISPLAY_BIN * 1000))
    cells = {}
    for k in range(len(lo)):
        key = ((lo[k] // d) * d, (lo[k] // d) * d + d) if hi[k] - lo[k] < d else (lo[k], hi[k])
        c = cells.setdefault(key, [0.0, 0.0, 0])
        c[0] += s[k]; c[1] += n[k]; c[2] += hi[k] - lo[k]
    bad = [k for k, c in cells.items() if c[2] != k[1] - k[0]]
    assert not bad, f"bins do not tile the {DISPLAY_BIN} display cells at {bad[:3]}"
    keys = sorted(k for k, c in cells.items() if c[1] > 0)
    tot = np.array([cells[k][1] for k in keys])
    return (np.array([(a + b) / 2000 for a, b in keys]),
            np.array([cells[k][0] for k in keys]) / tot, tot)


BAND_EDGES = sorted({e for r in OFFSET_BANDS.values() for e in r})


def segments(lo, hi):
    """[lo, hi] split at every band edge inside it, stopping just short of each
    cut so a step function is never evaluated in the next band."""
    cuts = [lo] + [e for e in BAND_EDGES if lo < e < hi] + [hi]
    return [(k0, k1 - 1e-9 if k1 < hi else k1)
            for k0, k1 in zip(cuts[:-1], cuts[1:]) if k1 - k0 > 1e-9]


def slide_data(path):
    """points, fits (polylines), h2_Unrel, and the coefficients, for one trait."""
    sums = load_sums(path)
    rows = []
    for classes, group, xoff in ((NOPO, "binned", 0.0), (["PO"], "parent_offspring", PO_XOFF)):
        a, m, n = display_bins(sums, classes)
        keep = (a <= SLIDE_MAX_A) & (n >= MIN_PAIRS_PER_BIN)
        rows.append(pd.DataFrame({"x": a[keep] + xoff, "y": m[keep],
                                  "n_pairs": n[keep].astype(int), "group": group}))
    points = pd.concat(rows, ignore_index=True)
    binned = points[points["group"] == "binned"]
    assert len(binned), f"{path}: no bin with >= {MIN_PAIRS_PER_BIN} pairs"
    x_lo, x_hi = binned["x"].min(), binned["x"].max()

    fits, coefs, seg = [], {}, 0
    for name, (ranges, design) in FITS.items():
        beta, a_fit = fit(sums, ranges, design)
        coefs[name] = beta
        if beta is None or not len(a_fit):
            continue
        lo, hi = max(a_fit.min(), x_lo), min(a_fit.max(), x_hi)
        for k0, k1 in (segments(lo, hi) if hi > lo else []):
            xs = np.linspace(k0, k1, 60)
            fits.append(pd.DataFrame({"model": name, "segment": seg, "x": xs, "y": design(xs) @ beta}))
            seg += 1
    beta_u, _ = fit(sums, *UNREL)
    h2u = float(beta_u[0]) if beta_u is not None else np.nan
    return points, pd.concat(fits, ignore_index=True), h2u, coefs


print(f"fits: {', '.join(FITS)}; dashed line: Unrel slope")

## Cross-check against `he_estimators` (optional)

If the repo is on this machine, the fits above are checked against what
`he_estimators` reports for the same merge. Without it this cell says so and
moves on — the notebook does not depend on it.

In [ ]:
import sys

_repo = next((p for p in (os.environ.get("AOUCOV_REPO", ""),
                          os.path.expanduser("~/repos/aou-covariance"),
                          os.path.expanduser("~/aou-covariance"))
              if p and os.path.isfile(os.path.join(p, "src", "he_estimators.py"))), None)
if _repo is None:
    print("he_estimators not found -- cross-check skipped")
else:
    sys.path.insert(0, os.path.join(_repo, "src"))
    import he_estimators as HE
    for r in SLIDES.itertuples():
        jk = r.full_tsv.replace(".full.tsv", ".jk.tsv")
        S, N = HE.load_arrays(r.full_tsv, jk, len(MID), HE.blocks_in(jk))
        est = HE.estimates(S, N, 0, MID)
        _, _, h2u, coefs = slide_data(r.full_tsv)
        read = lambda name, f: f(coefs[name]) if coefs[name] is not None else np.nan
        for key, ours in (("h2_Unrel", h2u),
                          ("b2_FS", read("FS", lambda b: 2 * b[1])),
                          ("h2_Rel", read("Rel", lambda b: b[0])),
                          ("h2_OneSlopeOffsets", read("OneSlopeOffsets", lambda b: b[0]))):
            theirs = est[(key, "noPO")]
            assert (np.isnan(ours) and np.isnan(theirs)) or np.isclose(
                ours, theirs, rtol=1e-9, atol=1e-12), (
                f"{r.phenotype}: {key} = {ours:.6g} here, {theirs:.6g} in he_estimators")
    print(f"matches he_estimators ({_repo}) for all {len(SLIDES)} traits")

## Render

Three build steps per trait in the UKB folder layout
(`wide/<step>/<stem>_<step>.png|pdf`), and the plotted values under `tables/`.

In [ ]:
VARIANTS = ["points_only", "points_h2", "with_fits"]
for v in VARIANTS:
    os.makedirs(f"{OUT_DIR}/wide/{v}", exist_ok=True)
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)


def slide_ylim(points, fits, ukb):
    """The UKB range, widened only on a side an All of Us point or fit would
    leave, with a 5% margin there. Returns (ylim, matched); None -> autoscale."""
    if ukb is None:
        return None, False
    ys = np.r_[points["y"].to_numpy(), fits["y"].to_numpy(), 0.0]
    lo, hi = float(ys.min()), float(ys.max())
    pad = 0.05 * (hi - lo)
    y0, y1 = ukb
    ylim = (y0 if lo >= y0 else lo - pad, y1 if hi <= y1 else hi + pad)
    return ylim, ylim == tuple(ukb)


params = []
for r in SLIDES.itertuples():
    points, fits, h2u, _ = slide_data(r.full_tsv)
    ylim, matched = slide_ylim(points, fits, r.ukb_ylim)
    points.to_csv(f"{OUT_DIR}/tables/{r.stem}_points.csv", index=False)
    fits.to_csv(f"{OUT_DIR}/tables/{r.stem}_fits.csv", index=False)
    params.append({"slide": r.slide, "phenotype": r.phenotype, "h2_unrel": h2u,
                   "ylim_lo": ylim[0] if ylim else np.nan,
                   "ylim_hi": ylim[1] if ylim else np.nan, "y_matches_ukb": matched})
    h2_line = h2u if np.isfinite(h2u) else None     # no line rather than a NaN line with a key
    for v in VARIANTS:
        panel(points, fits, v, f"{OUT_DIR}/wide/{v}/{r.stem}_{v}", h2_line, ylim)
    n_po = int((points["group"] == "parent_offspring").sum())
    yl = ("y = UKB" if matched else "own y" if ylim is None
          else f"y widened to ({ylim[0]:.2f}, {ylim[1]:.2f}), UKB ({r.ukb_ylim[0]:.2f}, {r.ukb_ylim[1]:.2f})")
    print(f"slide {r.slide}  {r.phenotype:<22} {len(points) - n_po:>3} bins, {n_po} PO, "
          f"smallest {points['n_pairs'].min():,} pairs | h2_Unrel {h2u:.3f} | {yl}")
    display(Image(filename=f"{OUT_DIR}/wide/with_fits/{r.stem}_with_fits.png", width=555))

pd.DataFrame(params).to_csv(f"{OUT_DIR}/tables/panel_params.csv", index=False)
print(f"\n-> {OUT_DIR}/wide/")

## Mirror locally

In [ ]:
# every PNG into one flat folder, to flick through without walking the bucket
os.makedirs(PLOTS_MIRROR, exist_ok=True)
_png = sorted(glob.glob(f"{OUT_DIR}/wide/**/*.png", recursive=True))
for _f in _png:
    shutil.copy2(_f, os.path.join(PLOTS_MIRROR, os.path.basename(_f)))
print(f"{len(_png)} PNG -> {PLOTS_MIRROR}")